# Phase 1: Environment Setup & Raw Schema Inspection

### Objective
Initialize the processing environment and set up memory-efficient data scanning for the LendingClub dataset using Polars LazyFrames.

### Technical Rationale
The raw LendingClub CSV is multi-gigabyte containing over 2 million rows and 100+ feature columns. Standard Pandas `read_csv` attempts to load the entire payload into RAM at once, leading to excessive memory pressure or out-of-memory (OOM) crashes.
- We use `pl.scan_csv` to establish a query plan to explore the dataset

In [1]:
import polars as pl

# Polars display config
pl.Config.set_tbl_rows(15)
pl.Config.set_tbl_cols(12)

# File path 
DATA_PATH = "/home/mike/Documents/work/DS/projects/credit-risk/data/raw/accepted_2007_to_2018Q4.csv"

# Step 1: Create the LazyFrame query graph
lazy_df = pl.scan_csv(
    DATA_PATH,
    infer_schema_length=10000,
    ignore_errors=True
)

In [4]:
lazy_df

In [6]:
schema = lazy_df.collect_schema()
total_columns = len(schema.names())

print(f"Total Columns Loaded: {total_columns}")

Total Columns Loaded: 151


In [7]:
lazy_df.head(5).collect()

id,member_id,loan_amnt,funded_amnt,funded_amnt_inv,term,…,debt_settlement_flag_date,settlement_status,settlement_date,settlement_amount,settlement_percentage,settlement_term
i64,str,f64,f64,f64,str,…,str,str,str,f64,f64,f64
68407277,null,3600.0,3600.0,3600.0,""" 36 months""",…,null,null,null,null,null,null
68355089,null,24700.0,24700.0,24700.0,""" 36 months""",…,null,null,null,null,null,null
68341763,null,20000.0,20000.0,20000.0,""" 60 months""",…,null,null,null,null,null,null
66310712,null,35000.0,35000.0,35000.0,""" 60 months""",…,null,null,null,null,null,null
68476807,null,10400.0,10400.0,10400.0,""" 60 months""",…,null,null,null,null,null,null


# Step 1.2: Data Sanitization, Target Engineering & Leakage Audit

### Objectives
1. **Schema Sanitization:** Coerce the `id` column to `pl.String` and drop malformed/corrupted rows (e.g., rogue header strings or footer metadata injected during CSV generation).
2. **Deterministic Target Definition:** Isolate historical loan outcomes into binary default indicator (`is_default`).
3. **Data Leakage Audit:** Strip out post-origination collection variables (e.g., `recoveries`, `total_rec_prncp`, `last_pymnt_d`) that contain future information inaccessible at loan underwriting time.

### Target Mapping Rules
- **Default (`1`):** `Charged Off`, `Default`, `Late (31-120 days)`, `Does not meet the credit policy. Status:Charged Off`
- **Non-Default (`0`):** `Fully Paid`, `Does not meet the credit policy. Status:Fully Paid`
- **Exclusions:** `Current`, `In Grace Period`, `Late (16-30 days)` (indeterminate performance window)

In [8]:
# Empirical Discovery: Scan unique loan statuses and their exact counts
loan_status_distribution = (
    lazy_df
    # Ensure ID is coerced to string and rogue rows are dropped first
    .with_columns(pl.col("id").cast(pl.String, strict=False).alias("id"))
    .filter(pl.col("id").is_not_null())
    .filter(pl.col("id").str.contains(r"^\d+$"))
    
    # Group and count occurrences across the entire raw dataset
    .group_by("loan_status")
    .agg(pl.len().alias("count"))
    .sort("count", descending=True)
    .collect()
)

print("Empirical Unique loan_status Distribution:")
print(loan_status_distribution)

Empirical Unique loan_status Distribution:
shape: (9, 2)
┌─────────────────────────────────┬─────────┐
│ loan_status                     ┆ count   │
│ ---                             ┆ ---     │
│ str                             ┆ u32     │
╞═════════════════════════════════╪═════════╡
│ Fully Paid                      ┆ 1076751 │
│ Current                         ┆ 878317  │
│ Charged Off                     ┆ 268559  │
│ Late (31-120 days)              ┆ 21467   │
│ In Grace Period                 ┆ 8436    │
│ Late (16-30 days)               ┆ 4349    │
│ Does not meet the credit polic… ┆ 1988    │
│ Does not meet the credit polic… ┆ 761     │
│ Default                         ┆ 40      │
└─────────────────────────────────┴─────────┘


In [11]:
#rows with text in the id column
total_raw_rows = lazy_df.select(pl.len()).collect().item()

sanitized_row_count = (
    lazy_df
    .with_columns(pl.col("id").cast(pl.String, strict=False).alias("id"))
    .filter(pl.col("id").is_not_null())
    .filter(pl.col("id").str.contains(r"^\d+$"))
    .select(pl.len())
    .collect()
    .item()
)

dropped_rows = total_raw_rows - sanitized_row_count
print(f"Total Raw Rows in CSV: {total_raw_rows:,}")
print(f"Rows After ID Sanitization: {sanitized_row_count:,}")
print(f"Dropped Malformed/Rogue Rows: {dropped_rows:,} ({(dropped_rows / total_raw_rows):.4%})")

Total Raw Rows in CSV: 2,260,701
Rows After ID Sanitization: 2,260,668
Dropped Malformed/Rogue Rows: 33 (0.0015%)


In [12]:
# Discover how loan statuses are distributed temporally across issue years
status_by_year_discovery = (
    lazy_df
    .with_columns(pl.col("id").cast(pl.String, strict=False).alias("id"))
    .filter(pl.col("id").is_not_null())
    .filter(pl.col("id").str.contains(r"^\d+$"))
    # Parse year from issue_d (format like Dec-2015)
    .with_columns(pl.col("issue_d").str.to_date("%b-%Y", strict=False).dt.year().alias("issue_year"))
    .group_by(["issue_year", "loan_status"])
    .agg(pl.len().alias("count"))
    .sort(["issue_year", "count"], descending=[False, True])
    .collect()
)

print(status_by_year_discovery)

shape: (60, 3)
┌────────────┬─────────────────────────────────┬────────┐
│ issue_year ┆ loan_status                     ┆ count  │
│ ---        ┆ ---                             ┆ ---    │
│ i32        ┆ str                             ┆ u32    │
╞════════════╪═════════════════════════════════╪════════╡
│ 2007       ┆ Does not meet the credit polic… ┆ 239    │
│ 2007       ┆ Fully Paid                      ┆ 206    │
│ 2007       ┆ Does not meet the credit polic… ┆ 113    │
│ 2007       ┆ Charged Off                     ┆ 45     │
│ 2008       ┆ Fully Paid                      ┆ 1315   │
│ 2008       ┆ Does not meet the credit polic… ┆ 582    │
│ 2008       ┆ Does not meet the credit polic… ┆ 249    │
│ 2008       ┆ Charged Off                     ┆ 247    │
│ …          ┆ …                               ┆ …      │
│ 2018       ┆ Current                         ┆ 427181 │
│ 2018       ┆ Fully Paid                      ┆ 47444  │
│ 2018       ┆ Charged Off                     ┆ 8867   │

# Step 1.3: Target Mapping & Temporal Out-of-Time (OOT) Split

### Objective
1. **Target Engineering:** Map our empirically verified statuses into a binary target (`is_default`).
   - `is_default = 1`: `Charged Off`, `Default`, `Late (31-120 days)`
   - `is_default = 0`: `Fully Paid`, along with legacy credit policy exceptions that resolved successfully.
   - *Exclusions:* Drop indeterminate active states (`Current`, `In Grace Period`, `Late (16-30 days)`).
2. **Temporal Partitioning:** Split the historical data before and after 2018 to create an Out-of-Time (OOT) validation set, preventing future data leakage.

In [13]:
# 1. Define empirical target mappings based on discovery
DEFAULT_STATUSES = [
    "Charged Off",
    "Default",
    "Late (31-120 days)",
    "Does not meet the credit policy. Status:Charged Off"
]

PAID_STATUSES = [
    "Fully Paid",
    "Does not meet the credit policy. Status:Fully Paid"
]
# 2. Execute target mapping and temporal feature extraction on the LazyFrame
modeled_lazy = (
    lazy_df
    # Sanitize ID
    .with_columns(pl.col("id").cast(pl.String, strict=False).alias("id"))
    .filter(pl.col("id").is_not_null())
    .filter(pl.col("id").str.contains(r"^\d+$"))
    
    # Filter to only include resolved states
    .filter(pl.col("loan_status").is_in(DEFAULT_STATUSES + PAID_STATUSES))
    
    # Create binary target
    .with_columns(
        pl.when(pl.col("loan_status").is_in(DEFAULT_STATUSES))
        .then(1)
        .otherwise(0)
        .alias("is_default")
    )
    
    # Extract issue year for temporal split
    .with_columns(
        pl.col("issue_d").str.to_date("%b-%Y", strict=False).dt.year().alias("issue_year")
    )
)

# 3. Collect into memory for partitioning
processed_df = modeled_lazy.collect()

# 4. Enforce OOT split (e.g., Development: < 2018, OOT Evaluation: >= 2018)
train_df = processed_df.filter(pl.col("issue_year") < 2018)
oot_df = processed_df.filter(pl.col("issue_year") >= 2018)

print(f"Total Processed Resolved Loans: {processed_df.height:,}")
print(f"In-Time Training Set (< 2018): {train_df.height:,} rows | Default Rate: {train_df['is_default'].mean():.2%}")
print(f"Out-of-Time Evaluation Set (>= 2018): {oot_df.height:,} rows | Default Rate: {oot_df['is_default'].mean():.2%}")

Total Processed Resolved Loans: 1,369,566
In-Time Training Set (< 2018): 1,306,027 rows | Default Rate: 21.04%
Out-of-Time Evaluation Set (>= 2018): 63,539 rows | Default Rate: 25.33%
